# មុខងារបែងចែកអត្ថបទ

ដូចដែលយើងបានរំលឹក អេភាសីយើងនឹងផ្តោតលើមុខងារបែងចែកអត្ថបទសាមញ្ញមួយ dựa trên **AG_NEWS** ភាពទិន្នន័យ ដែលជា​ការបែងចែកសេចក្តីដំបូន្មានព័ត៌មានទៅក្នុងមួយចំណាត់ថ្នាក់ពីរាបបួន៖ ពិភពលោក កីឡា ពាណិជ្ជកម្ម និង វិទ្យាសាស្រ្ត/បច្ចេកវិទ្យា។

## ព្រំប្រៃទិន្នន័យ

ព្រំប្រៃទិន្នន័យនេះត្រូវបានសាងសង់គ្នាចូលក្នុងម៉ូឌុល [`torchtext`](https://github.com/pytorch/text) ដូច្នេះ យើងអាចចូលដំណើរការយ៉ាងងាយស្រួល។


In [1]:
import torch
import torchtext
import os
import collections
os.makedirs('./data',exist_ok=True)
train_dataset, test_dataset = torchtext.datasets.AG_NEWS(root='./data')
classes = ['World', 'Sports', 'Business', 'Sci/Tech']

នៅទីនេះ `train_dataset` និង `test_dataset` មានប្រមាញ់ដែលត្រឡប់វ៉ាល្យូមនៃស្លាក (ចំនួនថ្នាក់) និងអត្ថបទដោយលំដាប់ ជាឧទាហរណ៍ៈ


In [2]:
list(train_dataset)[0]

(3,
 "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.")

ដូច្នេះ យើងចេញសារព័ត៌មានថ្មី ១០ ចំណងជើងដំបូងពីឃ្លាំងទិន្នន័យរបស់យើង៖


In [5]:
for i,x in zip(range(5),train_dataset):
    print(f"**{classes[x[0]]}** -> {x[1]}")


**Sci/Tech** -> Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
**Sci/Tech** -> Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.
**Sci/Tech** -> Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during the depth of the\summer doldrums.
**Sci/Tech** -> Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\flows from the main pipeline in southern Iraq after\intelligence showed a rebel militia could strike\infrastructure, an oil official said on Saturday.
**Sci/Tech** -> Oil prices soar to

ដោយសារតែទិន្នន័យជា iterator បើយើងចង់ប្រើទិន្នន័យជាច្រើនដងយើងត្រូវបម្លែងវាទៅជាបរមាណៈ list:


In [3]:
train_dataset, test_dataset = torchtext.datasets.AG_NEWS(root='./data')
train_dataset = list(train_dataset)
test_dataset = list(test_dataset)

## ការបំបែកពាក្យ

ឥឡូវនេះយើងត្រូវបំលែងអត្ថបទទៅជាខ្ទង់ **លេខ** ដែលអាចត្រូវបានតំណាងជាទិន្ស័រ។ ប្រសិនបើយើងចង់តំណាងជារាយនាមអក្សរពាក្យ ត្រូវធ្វើពីរជំហាន៖
* ប្រើ **tokenizer** ដើម្បីបំបែកអត្ថបទទៅជាប្រភេទពាក្យ **tokens**
* បង្កើត **vocabulary** សម្រាប់ពាក្យទាំងនោះ។


In [4]:
tokenizer = torchtext.data.utils.get_tokenizer('basic_english')
tokenizer('He said: hello')

['he', 'said', 'hello']

In [5]:
counter = collections.Counter()
for (label, line) in train_dataset:
    counter.update(tokenizer(line))
vocab = torchtext.vocab.vocab(counter, min_freq=1)

ប្រើវាក្យសព្ទ យើងអាចបំលែងខ្សែអក្សរដែលបានបំបែករបស់យើងទៅជាសំណុំលេខបានយ៉ាងងាយស្រួលៈ


In [19]:
vocab_size = len(vocab)
print(f"Vocab size if {vocab_size}")

stoi = vocab.get_stoi() # dict to convert tokens to indices

def encode(x):
    return [stoi[s] for s in tokenizer(x)]

encode('I love to play with my words')

Vocab size if 95810


[599, 3279, 97, 1220, 329, 225, 7368]

## ការបង្ហាញអត្ថបទជាថង់ពាក្យ

ដោយសារពាក្យសម្គាល់អត្ថន័យ ម្តងម្កាលយើងអាចរកឃើញអត្ថន័យនៃអត្ថបទតាមរយៈការមើលតែពាក្យនីមួយៗ ដោយមិនគិតពីលំដាប់របស់ពួកវានៅក្នុងប្រយោគ។ ឧទាហរណ៍ សម្រាប់ការតែងចំណាត់ថ្នាក់ព័ត៌មាន ពាក្យដូចជា *អាកាសធាតុ*, *ហេីយគ្រាប់ទឹកកក* មានសក្តានុពលបង្ហាញពី *ការព្យាករណ៍អាកាសធាតុ* ខណៈពាក្យដូចជា *ភាគហ៊ុន*, *ដុល្លារ* នឹងចាត់ទុកថាជា *ព័ត៌មានហិរញ្ញវត្ថុ*។

**ថង់ពាក្យ** (BoW) គឺជាការបង្ហាញវ៉ិចទ័រដែលប្រើប្រាស់ធម្មតាពិសេសច្រើនបំផុត។ ពាក្យនីមួយៗ ត្រូវបានភ្ជាប់ទៅនឹងឧSobatាយវ៉ិចទ័រ ដោយធាតុវ៉ិចទ័រមានចំនួនករណីកើតឡើងនៃពាក្យនៅក្នុងឯកសារដែលបានផ្ដល់។

![រូបភាពបង្ហាញពីរបៀបដែលការបង្ហាញវ៉ិចទ័រថង់ពាក្យត្រូវបានផ្ទុកនៅក្នុងអង្គចងចាំ។](../../../../../translated_images/km/bag-of-words-example.606fc1738f1d7ba9.webp) 

> **សំគាល់**: អ្នកគួរតែគិតថា BoW ជា៉ការសរុបវ៉ិចទ័រ one-hot-encoded សម្រាប់ពាក្យនីមួយៗនៅក្នុងអត្ថបទ។

​ខាង​ក្រោម​ជា​ឧទាហរណ៍​នៃ​របៀប​បង្កើត​ការ​បង្ហាញ​ថង់ពាក្យ​ប្រើ​ប្រាស់​បណ្ណាល័យ Scikit Learn ក្នុងភាសា python ៖


In [7]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
corpus = [
        'I like hot dogs.',
        'The dog ran fast.',
        'Its hot outside.',
    ]
vectorizer.fit_transform(corpus)
vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()

array([[1, 1, 0, 2, 0, 0, 0, 0, 0]], dtype=int64)

ដើម្បីគណនាពុម្ពទំនុកចិត្ត bag-of-words ពីតំណាងវ៉ិកទ័រនៃឃ្លាសន្និដ្ឋាន AG_NEWS របស់យើង យើងអាចប្រើមុខងារដូចខាងក្រោម៖


In [20]:
vocab_size = len(vocab)

def to_bow(text,bow_vocab_size=vocab_size):
    res = torch.zeros(bow_vocab_size,dtype=torch.float32)
    for i in encode(text):
        if i<bow_vocab_size:
            res[i] += 1
    return res

print(to_bow(train_dataset[0][1]))

tensor([2., 1., 2.,  ..., 0., 0., 0.])


> **កំណត់សម្គាល់:** នៅទីនេះយើងកំពុងប្រើអថេរ `vocab_size` កម្រិតសកល ដើម្បីបញ្ជាក់ទំហំលំនាំដើមនៃវាក្យសព្ទ។ ពីព្រោះទំហំពាក្យសព្ទជាញែកខ្លាំងជារឿយៗ យើងអាចកំណត់ទំហំវាក្យសព្ទទៅតែពាក្យដែលមានភាពញឹកញាប់ខ្ពស់បំផុត។ សូមព្យាយាមកាត់បន្ថយតម្លៃ `vocab_size` ហើយរត់កូដខាងក្រោម ហើយមើលថាវាមានឥទ្ធិពលយ៉ាងណាលើភាពត្រឹមត្រូវ។ អ្នកគួរត្រូវរំពឹងថា ភាពត្រឹមត្រូវនឹងធ្លាក់ចុះបន្តិច ប៉ុន្តែមិនខ្លាំងទេ ដើម្បីសំរាប់ការសមត្ថភាពខ្ពស់ជាងមុន។


## ការបណ្តុះបណ្តាលអ្នកចាត់ថ្នាក់ BoW

ឥឡូវនេះដែលយើងបានរៀនថាតើធ្វើដូចម្តេចដើម្បីបង្កើតតំណាងបង្គោលពាក្យ Bag-of-Words នៃអត្ថបទរបស់យើងហើយ យើងចង់បណ្តុះបណ្តាលអ្នកចាត់ថ្នាក់សម្រាប់វា។ ជាមួយនេះ ដំបូង យើងត្រូវបម្លែងទិន្នន័យសម្រាប់ការបណ្តុះបណ្តាលដោយរបៀបគឺ ដំណាក់កាលតំណាងពាក្យតាមទីតាំងទាំងអស់ត្រូវបានបម្លែងទៅជាតំណាង Bag-of-Words។ នេះអាចសម្រេចបានដោយផ្ទេរអនុគមន៍ `bowify` ជាព៉ារ៉ាម៉ែត្រ `collate_fn` ទៅឲ្យ `DataLoader` នៃ torch ដែលមានស្តង់ដារ៖


In [21]:
from torch.utils.data import DataLoader
import numpy as np 

# this collate function gets list of batch_size tuples, and needs to 
# return a pair of label-feature tensors for the whole minibatch
def bowify(b):
    return (
            torch.LongTensor([t[0]-1 for t in b]),
            torch.stack([to_bow(t[1]) for t in b])
    )

train_loader = DataLoader(train_dataset, batch_size=16, collate_fn=bowify, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, collate_fn=bowify, shuffle=True)

ឥឡូវនេះចូរអនុស្សាណារៀបចំបណ្តាញសរសេរមួយសម្រាប់ចំណាត់ថ្នាក់សាមញ្ញ ដែលមានស្រទាប់ខាងស្រទាប់តែមួយ។ ទំហំវ៉ិចទ័រចូលមានទំហំស្មើនឹង `vocab_size` ហើយទំហំចេញឆបគ្នានឹងចំនួនថ្នាក់ (4)។ ពីព្រោះយើងកំពុងដោះស្រាយបញ្ហាចំណាត់ថ្នាក់ ការជំរុញចុងក្រោយគឺជា `LogSoftmax()`។


In [22]:
net = torch.nn.Sequential(torch.nn.Linear(vocab_size,4),torch.nn.LogSoftmax(dim=1))

ឥឡូវនេះយើងនឹងកំណត់រង្វិលបណ្តុះបណ្តាល PyTorch ស្តង់ដារ។ ព្រោះតែកំណត់ទិន្នន័យរបស់យើងមានទំហំធំជាងមធ្យម សម្រាប់គោលបំណងបង្រៀនយើងនឹងបណ្តុះបណ្តាលគ្រាន់តែកំណត់មួយទេ ហើយខ្លះពេលម្យ៉ាងទៀតក៏អាចបណ្តុះបណ្តាលតិចជាងមួយកំណត់ផង (ការ​បញ្ជាក់​ពាក្យបញ្ជា `epoch_size` អនុញ្ញាតឲ្យយើងកំណត់ដែនកំណត់ការបណ្តុះបណ្តាល)។ យើងក៏នឹងរាយការណ៍ភាពត្រឹមត្រូវបណ្តុះបណ្តាលដែលបានសង្ស័យក្នុងអំឡុងពេលបណ្តុះបណ្តាលផងដែរ; ភាគរយនៃការរាយការណ៍ត្រូវបានកំណត់ដោយការបញ្ជាក់ពាក្យបញ្ជា `report_freq`។


In [24]:
def train_epoch(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.NLLLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,features in dataloader:
        optimizer.zero_grad()
        out = net(features)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count

In [25]:
train_epoch(net,train_loader,epoch_size=15000)

3200: acc=0.8028125
6400: acc=0.8371875
9600: acc=0.8534375
12800: acc=0.85765625


(0.026090790722161722, 0.8620069296375267)

## BiGrams, TriGrams និង N-Grams

ការកំណត់មួយនៃវិធីសាស្រ្តថតពាក្យ (bag of words) គឺពាក្យខ្លះជាផ្នែកមួយនៃការបង្កើតអត្ថាធិប្បាយពាក្យច្រើនពាក្យ ដូចជាឧទាហរណ៍ ពាក្យ 'hot dog' មានន័យខុសពីពាក្យ 'hot' និង 'dog' ក្នុងបរិបទផ្សេងៗ។ ប្រសិនបើយើងបង្ហាញពាក្យ 'hot' និង 'dog' ជានិរន្តរភាពដោយវ៉ិចទ័រដូចគ្នា វាអាចបង្កការជ្រុះច្រឡំចំពោះម៉ូដែលរបស់យើង។

ដើម្បីដោះស្រាយបញ្ហានេះ, **ការតំណាង N-gram** ត្រូវបានប្រើជាញឹកញាប់ក្នុងវិធីសាស្រ្តចាត់ថ្នាក់ឯកសារ, ដែលប្រេកង់នៃពាក្យរាល់ខ្ទង់ពាក្យពីរឬបីពាក្យគឺជាលក្ខណៈសំខាន់សំរាប់ការបណ្តុះបណ្តាលម៉ាស៊ីនចាត់ថ្នាក់។ ក្នុងការតំណាង bigram, ឧទាហរណ៍យើងនឹងបន្ថែមដំណាក់កាលពាក្យទាំងអស់ទៅក្នុងវ៉ុកាប្យូលារី ខៀវបន្ថែមពីពាក្យដើម។

ខាងក្រោមជាឧទាហរណ៍ពីរបៀបបង្កើតការតំណាង bigram bag of word ដោយប្រើ Scikit Learn៖


In [26]:
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2), token_pattern=r'\b\w+\b', min_df=1)
corpus = [
        'I like hot dogs.',
        'The dog ran fast.',
        'Its hot outside.',
    ]
bigram_vectorizer.fit_transform(corpus)
print("Vocabulary:\n",bigram_vectorizer.vocabulary_)
bigram_vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()


Vocabulary:
 {'i': 7, 'like': 11, 'hot': 4, 'dogs': 2, 'i like': 8, 'like hot': 12, 'hot dogs': 5, 'the': 16, 'dog': 0, 'ran': 14, 'fast': 3, 'the dog': 17, 'dog ran': 1, 'ran fast': 15, 'its': 9, 'outside': 13, 'its hot': 10, 'hot outside': 6}


array([[1, 0, 1, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
      dtype=int64)

ចំណុចខ្វះខាតសំខាន់នៃវិធីសាស្រ្ត N-gram គឺទំហំវេយ្យាករណ៍ចាប់ផ្តើមរីកចម្រើនយ៉ាងឆាប់រហ័ស ។ ក្នុងការអនុវត្តន៍ ពួកយើងត្រូវការចងក្រងការ​តំណាង N-gram ជាមួយបច្ចេកទេស​ការកាត់បន្ថយ​រាងវេកទិសខ្លះៗ ដូចជា *embeddings* ដែលយើងនឹងពិភាក្សានៅឯកតាចុងក្រោយ។

ដើម្បីប្រើប្រាស់ការតំណាង N-gram នៅក្នុងទិន្នន័យ **AG News** របស់យើង យើងត្រូវតែបង្កើតវេយ្យាករណ៍ ngram ពិសេស ៖


In [27]:
counter = collections.Counter()
for (label, line) in train_dataset:
    l = tokenizer(line)
    counter.update(torchtext.data.utils.ngrams_iterator(l,ngrams=2))
    
bi_vocab = torchtext.vocab.vocab(counter, min_freq=1)

print("Bigram vocabulary length = ",len(bi_vocab))

Bigram vocabulary length =  1308842


យើងអាចប្រើកូដដូចខាងលើដើម្បីបណ្តុះម៉ាស៊ីនចាត់ថ្នាក់ ប៉ុន្តែវានឹងប្រើអង្គចងចាំខ្លាំងណាស់។ នៅឯកត្តាបន្ទាប់ យើងនឹងបណ្តុះម៉ាស៊ីនចាត់ថ្នាក់ bigram ដោយប្រើ embeddings។

> **សម្គាល់:** អ្នកអាចទុកតែ ngrams ដែលបញ្ចេញឡើងក្នុងអត្ថបទច្រើនជាងចំនួនដែលបានបញ្ជាក់។ វានឹងធានាថា bigrams ដែលមានកំណត់នឹងត្រូវដកចេញ ហើយនឹងបន្ថយវិមាត្រជាប្រសើរណាស់។ ដើម្បីធ្វើដូចនេះ សូមកំណត់ប៉ារ៉ាម៉ែត្រ `min_freq` ទៅតម្លៃខ្ពស់ជាងនេះ ហើយមើលការផ្លាស់ប្តូរវែងនៃវចនានុក្រម។


## Term Frequency Inverse Document Frequency TF-IDF

ក្នុងការតំណាង BoW, ការបង្ហាញពាក្យត្រូវបានគិតសោយាក់កម្រ, មិនគិតថាពាក្យនោះជាពាក្យអ្វីនោះទេ។ ទោះជាយ៉ាងណា, គឺច្បាស់ថាពាក្យដែលមានប្រេកង់ច្រើន ដូចជា *a*, *in* ល។ មានសារៈសំខាន់តិចជាងបច្ចេកទេសពិសេស។ ជាការពិត, ក្នុងភារកិច្ច NLP ភាគច្រើនពាក្យខ្លះមានទំនាក់ទំនងសំខាន់ជាងពាក្យផ្សេងទៀត។

**TF-IDF** មានន័យថា **term frequency–inverse document frequency**។ វាជាបម្លែងពីបាវពាក្យដែលជំនួសដោយតម្លៃពីរបៀប 0/1 ដែលបង្ហាញពីការបង្ហាញរបស់ពាក្យមួយក្នុងឯកសារ ដែលមានតម្លៃជាចំនួនទសភាគ ដែលទាក់ទងនឹងប្រេកង់នៃការបង្ហាញពាក្យនៅក្នុងក្រុមឯកសារ។

យ៉ាងជាក់លាក់, ទម្ងន់ $w_{ij}$ នៃពាក្យ $i$ នៅក្នុងឯកសារ $j$ ត្រូវបានកំណត់ដូចជា៖
$$
w_{ij} = tf_{ij}\times\log({N\over df_i})
$$
ដែល
* $tf_{ij}$ គឺជាចំនួនការបង្ហាញនៃ $i$ នៅក្នុង $j$, គឺតម្លៃ BoW ដែលយើងបានឃើញមុន
* $N$ គឺជាចំនួនឯកសារ​នៅក្នុងប្រមូលផ្តុំ
* $df_i$ គឺជាចំនួនឯកសារដែលមានពាក្យ $i$ នៅក្នុងប្រមូលផ្តុំទាំងមូល

តម្លៃ TF-IDF $w_{ij}$ បន្ថែមឡើងជាផ្តាច់មុខទៅនឹងចំនួនដងដែលពាក្យមួយប្រែប្រួលក្នុងឯកសារ ហើយត្រូវបានពន្លាក់ដោយចំនួនឯកសារនៅក្នុងក្រុមឯកសារដែលមានពាក្យនោះ ដែលជួយកែសម្រួលសម្រាប់ការពិតថាពាក្យខ្លះបង្ហាញញឹកញាប់ជាងពាក្យផ្សេងៗ។ ឧទាហរណ៍ ប្រសិនបើពាក្យបង្ហាញនៅក្នុង *គ្រប់* ឯកសារនៅក្នុងប្រមូលផ្តុំ, $df_i=N$, ហើយ $w_{ij}=0$, ហើយពាក្យទាំងនោះនឹងត្រូវបានមិនគិតទេ។

អ្នកអាចបង្កើតបច្ចេកទេសវ៉េកទ័រ TF-IDF របស់អត្ថបទដោយងាយស្រួលជាមួយ Scikit Learn:


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1,2))
vectorizer.fit_transform(corpus)
vectorizer.transform(['My dog likes hot dogs on a hot day.']).toarray()

array([[0.43381609, 0.        , 0.43381609, 0.        , 0.65985664,
        0.43381609, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ]])

## និទស្សន៏

ទោះបីជាការបង្ហាញ TF-IDF ផ្តល់ទំងន់ប្រេកង់ចំពោះពាក្យផ្សេងៗក៏ដោយ វាមិនអាចបង្ហាញអត្ថន័យឬលំដាប់បានទេ។ ដូចដែលវចនករល្បី J. R. Firth បាននិយាយនៅឆ្នាំ ១៩៣៥ ថា “អត្ថន័យពេញលេញនៃពាក្យមួយគឺជានិរិច្ឆេទតែប៉ុណ្ណោះ ហើយការសិក្សាអត្ថន័យដោយមិនពាក់ព័ន្ធនឹងបរិបទមិនអាចយកត្រឹមត្រូវបានទេ។”។ យើងនឹងរៀនបន្ថែមនៅពេលក្រោយក្នុងវគ្គថ្នាក់របៀបចាប់យកព័ត៌មានបរិបទពីអត្ថបទដោយប្រើគំរូភាសា។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការពិបាក**៖
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលដែលយើងខិតខំសម្រាប់ភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថាបកប្រែដោយស្វ័យប្រវត្តិក្នុងខ្លះអាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដែលមានភាសាមូលដ្ឋានគួរត្រូវបានគេចាត់ទុកជាផ្លូវការមួយ។ សម្រាប់ព័ត៌មានសំខាន់ៗ ការបកប្រែដោយអ្នកជំនាញមនុស្សគឺមានការផ្តល់អនុសាសន៍។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំឬការបកស្រាយខុសឆ្គងដែលកើតឡើងពីការប្រើប្រាស់បកប្រែនេះឡើយ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
